# Exemplo de uso dos SHAP Values

Fonte: [Site oficial SHAP Values](https://shap.readthedocs.io/en/latest/index.html) 

In [ ]:
import matplotlib.pylab as plt
import numpy as np
import xgboost
from sklearn.model_selection import train_test_split
import math
import shap

shap.initjs()

#### Carregar o Dataframe

**DataFrame: Adult census**

Age (float) : Age in years.

Workclass (category) : Type of employment.

fnlwgt (float) : Final weight; the number of units in the target population that the record represents.

Education (category) : Highest level of education achieved.

Education-Num (float) : Numeric representation of education level.

Marital Status (category) : Marital status of the individual.

Occupation (category) : Type of occupation.

Relationship (category) : Relationship status.

Race (category) : Ethnicity of the individual.

Sex (category) : Gender of the individual.

Capital Gain (float) : Capital gains recorded.

Capital Loss (float) : Capital losses recorded.

Hours per week (float) : Number of hours worked per week.

Country (category) : Country of origin.

Target (category) : Binary target variable indicating whether the individual earns more than 50K.

In [ ]:
X, y = shap.datasets.adult()
X_display, y_display = shap.datasets.adult(display=True)
X

In [ ]:
# create a train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

In [ ]:
# Create DMatrixs
d_train = xgboost.DMatrix(X_train, label=y_train)
d_test = xgboost.DMatrix(X_test, label=y_test)

#### Treinar o modelo

In [ ]:
params = {
    "eta": 0.01, # learning rate
    "objective": "binary:logistic", # objective function
    "subsample": 0.5, # subsample ratio of the training instances
    "eval_metric": "logloss", # evaluation metric
}
model = xgboost.train(
    params,
    d_train,
    num_boost_round = 5000, 
    evals=[(d_test, "test")],
    verbose_eval=100,
    early_stopping_rounds=20, 
)

#### Explicar as predições do modelo usando SHAP Values

In [ ]:
# Create object that can calculate SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

In [ ]:
sample_index = np.random.randint(0, X.shape[0])

print(f"Sample_index: {sample_index}, expected_value: {y_display[sample_index]} ")
print(f"Model output raw: {model.predict(xgboost.DMatrix(X.iloc[[sample_index]]),output_margin=True)}")
print(f"Model output sigmoid: {model.predict(xgboost.DMatrix(X.iloc[[sample_index]]))}")

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[sample_index, :], X_display.iloc[sample_index])

In [ ]:
shap.summary_plot(shap_values, X)

In [ ]:
shap.summary_plot(shap_values, X_display, plot_type="bar")

# Bonus - Comparar Algoritmos de Feature Importance: 

# Mutual Information (Information Gain) Gain vs SHAP Values

## O que é Information Gain (Ganho de Informação)?

**Information Gain** é uma métrica usada em árvores de decisão que mede quanto uma feature reduz a incerteza (entropia) sobre a variável alvo.

### Quando usar:
- Análise rápida de importância global
- Interpretabilidade local limitada

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd


# Calcular Mutual Information (information gain)
mi_scores = mutual_info_classif(X, y, random_state=42)


# Calcular SHAP (já calculado)
shap_scores = np.abs(shap_values).mean(axis=0)

# Criar dataframe de comparação
comparison = pd.DataFrame({
    'Feature': X.columns,
    'Mutual_Information': mi_scores,
    'SHAP_Mean': shap_scores
})

# Normalizar para escala 0-1
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
comparison[['MI_norm', 'SHAP_norm']] = scaler.fit_transform(
    comparison[['Mutual_Information', 'SHAP_Mean']]
)

comparison = comparison.sort_values('Mutual_Information', ascending=False)

print("\nRanking de Importância (Top 10):")
print(comparison[['Feature', 'MI_norm', 'SHAP_norm']].head(10).round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(comparison.head(10)))
width = 0.25

ax.bar(x_pos, comparison['MI_norm'].head(10), width, label='Mutual Information', alpha=0.8)
ax.bar(x_pos + width, comparison['SHAP_norm'].head(10), width, label='SHAP Values', alpha=0.8)

ax.set_xlabel('Features', fontsize=11)
ax.set_ylabel('Importância (Normalizada)', fontsize=11)
ax.set_title('Comparação: MI vs SHAP', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(comparison['Feature'].head(10), rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
